# 01 - Document Loading & Global Metadata Tagging

**Phase 1, Step 1** of the Ingestion & Chunking Lifecycle.

**Objective:** Load documents in multiple formats (PDF, DOCX, TXT, XLSX) and assign global metadata fields for RBAC filtering.

### Metadata Schema

| Field | Scope | Description |
|---|---|---|
| `clearance_level` | Global | 0=public, 1=intern, 2=confidential, 3=strict |
| `allowed_departments` | Global | List of departments (applies when clearance >= 2) |
| `source_file` | Global | Original filename |
| `doc_type` | Global | report, manual, contract, log, spreadsheet |
| `global_topic` | Global | High-level topic description |
| `chunk_id` | Local (Phase 2) | Unique chunk identifier |
| `page_number` | Local (Phase 2) | Source page reference |
| `contains_PII` | Local (Phase 2) | Boolean for sensitive personal data |

In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.schema import Document
from docx import Document as DocxDocument
import pandas as pd
import os
from typing import Optional

DOCS_DIR = "../../../data/raw_docs/"

## 1. Multi-Format Document Loaders

Each loader returns a `list[Document]` with LangChain's Document schema, ensuring a uniform interface regardless of source format.

In [2]:
def load_pdf(filepath: str) -> list[Document]:
    """Load PDF using PyPDFLoader. Returns one Document per page."""
    loader = PyPDFLoader(filepath)
    pages = loader.load()
    # Normalize metadata key: PyPDFLoader uses 'page' (0-indexed)
    for p in pages:
        p.metadata["page_number"] = p.metadata.pop("page", 0) + 1
    return pages


def load_docx(filepath: str) -> list[Document]:
    """Load DOCX using python-docx. Returns one Document with full text."""
    doc = DocxDocument(filepath)
    full_text = "\n".join(p.text for p in doc.paragraphs if p.text.strip())
    return [Document(page_content=full_text, metadata={"source": filepath, "page_number": 1})]


def load_txt(filepath: str) -> list[Document]:
    """Load plain text file. Returns one Document."""
    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read()
    return [Document(page_content=content, metadata={"source": filepath, "page_number": 1})]


def load_xlsx(filepath: str) -> list[Document]:
    """Load Excel spreadsheet. Converts each sheet to text representation."""
    xls = pd.ExcelFile(filepath)
    docs = []
    for i, sheet_name in enumerate(xls.sheet_names):
        df = pd.read_excel(xls, sheet_name=sheet_name)
        text = f"Sheet: {sheet_name}\n{df.to_string(index=False)}"
        docs.append(Document(
            page_content=text,
            metadata={"source": filepath, "page_number": i + 1, "sheet_name": sheet_name}
        ))
    return docs


# Dispatcher by file extension
LOADERS = {
    ".pdf": load_pdf,
    ".docx": load_docx,
    ".txt": load_txt,
    ".xlsx": load_xlsx,
}


def load_document(filepath: str) -> list[Document]:
    """Detect format and load document into LangChain Documents."""
    ext = os.path.splitext(filepath)[1].lower()
    loader_fn = LOADERS.get(ext)
    if loader_fn is None:
        raise ValueError(f"Unsupported file format: {ext}")
    return loader_fn(filepath)


print("Loaders registered for:", list(LOADERS.keys()))

Loaders registered for: ['.pdf', '.docx', '.txt', '.xlsx']


## 2. Global Metadata Tagging

Assigns RBAC metadata to every page/section of a loaded document. In production, an LLM agent would infer these fields automatically; here we simulate manual assignment to focus on the chunking pipeline.

In [3]:
CLEARANCE_MAP = {"public": 0, "intern": 1, "confidential": 2, "strict": 3}


def apply_global_metadata(
    docs: list[Document],
    source_file: str,
    doc_type: str,
    global_topic: str,
    clearance: str,
    department: Optional[str] = None,
) -> list[Document]:
    """Apply global metadata fields to all documents from a single source."""
    if clearance not in CLEARANCE_MAP:
        raise ValueError(f"Invalid clearance '{clearance}'. Must be one of {list(CLEARANCE_MAP.keys())}")

    level = CLEARANCE_MAP[clearance]

    for doc in docs:
        doc.metadata["source_file"] = source_file
        doc.metadata["doc_type"] = doc_type
        doc.metadata["global_topic"] = global_topic
        doc.metadata["clearance_level"] = level
        # Department filtering only applies at clearance >= 2
        doc.metadata["allowed_departments"] = department if level >= 2 else "all"

    return docs

## 3. Load & Tag All Test Documents

Define the document registry with their known metadata, then load and tag each one.

In [4]:
# Document registry: each entry defines the known metadata for a test file
DOCUMENT_REGISTRY = [
    {
        "filename": "Witty-QuickGuide-EN.pdf",
        "doc_type": "manual",
        "global_topic": "Witty Timer hardware quick start guide",
        "clearance": "public",
        "department": None,
    },
    {
        "filename": "Witty-Financial-Report-2025.pdf",
        "doc_type": "report",
        "global_topic": "Witty product line Q3 2025 financial performance",
        "clearance": "strict",
        "department": "finance",
    },
    {
        "filename": "distribution-contract-2026.docx",
        "doc_type": "contract",
        "global_topic": "Exclusive distribution contract for Witty Timer in Spain",
        "clearance": "confidential",
        "department": "legal",
    },
    {
        "filename": "server_logs_witty_backend.txt",
        "doc_type": "log",
        "global_topic": "Witty Manager API backend server logs",
        "clearance": "confidential",
        "department": "engineering",
    },
    {
        "filename": "clients-and-billings.xlsx",
        "doc_type": "spreadsheet",
        "global_topic": "Client database with billing and support tier info",
        "clearance": "strict",
        "department": "sales",
    },
]

# Load and tag all documents
all_documents: dict[str, list[Document]] = {}

for entry in DOCUMENT_REGISTRY:
    filepath = os.path.join(DOCS_DIR, entry["filename"])
    try:
        docs = load_document(filepath)
        docs = apply_global_metadata(
            docs,
            source_file=entry["filename"],
            doc_type=entry["doc_type"],
            global_topic=entry["global_topic"],
            clearance=entry["clearance"],
            department=entry["department"],
        )
        all_documents[entry["filename"]] = docs
        print(f"[OK] {entry['filename']}: {len(docs)} section(s) loaded")
    except Exception as e:
        print(f"[ERROR] {entry['filename']}: {e}")

print(f"\nTotal documents loaded: {len(all_documents)}")

[OK] Witty-QuickGuide-EN.pdf: 2 section(s) loaded


[OK] Witty-Financial-Report-2025.pdf: 2 section(s) loaded
[OK] distribution-contract-2026.docx: 1 section(s) loaded
[OK] server_logs_witty_backend.txt: 1 section(s) loaded
[OK] clients-and-billings.xlsx: 1 section(s) loaded

Total documents loaded: 5


## 4. Inspection: Metadata & Content Preview

In [5]:
for filename, docs in all_documents.items():
    print(f"\n{'='*70}")
    print(f"FILE: {filename}")
    print(f"Sections: {len(docs)}")
    print(f"Metadata: {docs[0].metadata}")
    preview = docs[0].page_content[:300].replace('\n', ' ')
    print(f"Content preview: {preview}...")
    print(f"Total characters: {sum(len(d.page_content) for d in docs)}")


FILE: Witty-QuickGuide-EN.pdf
Sections: 2
Metadata: {'source': '../data/raw_docs/Witty-QuickGuide-EN.pdf', 'page_number': 1, 'source_file': 'Witty-QuickGuide-EN.pdf', 'doc_type': 'manual', 'global_topic': 'Witty Timer hardware quick start guide', 'clearance_level': 0, 'allowed_departments': 'all'}
Content preview: ENG QUICK GUIDE Witty Manager Software The USB stick contains the Witty Manager software, as well as the relevant user manual, which you should open or print. To execute the program, the PC must run Windows OS (XP/Vista/7/8). The steps for installing (chapter 2) and  using the software (chapter 3) a...
Total characters: 2248

FILE: Witty-Financial-Report-2025.pdf
Sections: 2
Metadata: {'source': '../data/raw_docs/Witty-Financial-Report-2025.pdf', 'page_number': 1, 'source_file': 'Witty-Financial-Report-2025.pdf', 'doc_type': 'report', 'global_topic': 'Witty product line Q3 2025 financial performance', 'clearance_level': 3, 'allowed_departments': 'finance'}
Content preview: M

## 5. Export for Notebook 02

Serialize the loaded documents so the chunking notebook can import them directly.

In [6]:
import json

# Serialize to JSON for cross-notebook use
serialized = {}
for filename, docs in all_documents.items():
    serialized[filename] = [
        {"page_content": d.page_content, "metadata": d.metadata}
        for d in docs
    ]

output_path = "../../../data/results/notebook_results/loaded_documents.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(serialized, f, ensure_ascii=False, indent=2)

print(f"Exported {len(serialized)} documents to {output_path}")

Exported 5 documents to ../data/loaded_documents.json
